# LF Whale Hunter

Look up Polymarket proxy wallet addresses from a list of usernames,
fetch their trade history, and produce a `WhaleAnalysis` for each.

In [ ]:
from src.trading_tools.apps.whale_monitor.analyser import analyse_markets

%load_ext autoreload
%autoreload 2

import logging
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv(Path("../../..").resolve() / ".env")

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("httpx").setLevel(logging.WARNING)
print("Setup complete.")

## Configuration

In [ ]:
INPUT_CSV = Path("whales_to_watch/lf_whales_input.csv")
MAX_TRADE_PAGES = 10  # pages of 500 trades per wallet (increase for deeper history)
ENRICH = True  # fetch Gamma market metadata for each trade (cached per market)

## Step 1 — Load usernames from CSV

In [ ]:
users_df = pd.read_csv(INPUT_CSV)
usernames = users_df["username"].tolist()
print(f"Loaded {len(usernames)} usernames: {usernames}")

## Step 2 — Resolve usernames to proxy wallet addresses

Uses `PolymarketClient.lookup_user_address()` which queries the
`/v1/leaderboard?userName=` endpoint.

In [ ]:
from trading_tools.clients.polymarket.client import PolymarketClient

# Use cached addresses from the CSV where available; fall back to API lookup.
address_map: dict[str, str | None] = {}
cached = dict(
    zip(users_df["username"], users_df.get("address", [None] * len(users_df)), strict=False)
)

to_lookup = [u for u, a in cached.items() if pd.isna(a) or not a]

if to_lookup:
    async with PolymarketClient() as client:
        for username in to_lookup:
            cached[username] = await client.lookup_user_address(username)

address_map = {u: (None if pd.isna(a) else str(a)) for u, a in cached.items()}

found = {u: a for u, a in address_map.items() if a}
missing = [u for u, a in address_map.items() if not a]

for u, a in address_map.items():
    print(f"  {u:20s} -> {a or 'NOT FOUND'}")
print(f"\nResolved {len(found)}/{len(usernames)} usernames.")
if missing:
    print(f"Not found: {missing}")

## Step 3 — Collect trade history

Fetches raw trades from `/trades?user=<address>` and converts each dict to a
`WhaleTrade` instance via `_parse_trade()`.

In [ ]:
from trading_tools.apps.whale_monitor.collector import _parse_trade
from trading_tools.apps.whale_monitor.enricher import MarketMetadata, enrich_trades
from trading_tools.clients.polymarket.client import PolymarketClient

_TRADES_PER_PAGE = 500

# username -> list[WhaleTrade] (plain) or list[EnrichedTrade] (when ENRICH=True)
whale_trades: dict[str, list] = {}

# Shared enrichment cache — condition IDs resolved here are not re-fetched
# across traders, so the API is called at most once per unique market.
enrich_cache: dict[str, MarketMetadata] = {}

async with PolymarketClient() as client:
    for username, address in found.items():
        try:
            raw_trades = []
            now_ms = int(time.time() * 1000)

            for page_num in range(MAX_TRADE_PAGES):
                page = await client.get_trader_trades(
                    address,
                    limit=_TRADES_PER_PAGE,
                    offset=page_num * _TRADES_PER_PAGE,
                )
                raw_trades.extend(page)
                if len(page) < _TRADES_PER_PAGE:
                    break

            trades = [
                t for raw in raw_trades if (t := _parse_trade(raw, address, now_ms)) is not None
            ]

            if ENRICH:
                trades = await enrich_trades(client, trades, cache=enrich_cache)

            whale_trades[username] = trades
            print(f"{username}: {len(trades)} trades collected" + (" (enriched)" if ENRICH else ""))
        except Exception as e:
            logging.debug(f"Error fetching trades for {username} ({address}): {e}")

if ENRICH:
    print(f"\nEnrichment cache: {len(enrich_cache)} unique markets resolved")

## Step 4 — Analyse

Runs `analyse_trades()` over each entry in `whale_trades`.

In [ ]:
from trading_tools.apps.whale_monitor.analyser import analyse_trades

trade_analysis = {
    username: analyse_trades(found[username], trades) for username, trades in whale_trades.items()
}

market_analysis = {username: analyse_markets(trades) for username, trades in whale_trades.items()}

for username, analysis in trade_analysis.items():
    print(
        f"{username}: {analysis.total_trades} trades, "
        f"volume=${analysis.total_volume:,.0f}, "
        f"{analysis.unique_markets} markets"
    )

## Step 5 — Inspect results

In [ ]:
whale_trades["xdd07070"][0]

## Step 5 — summarize_trades + breakdown functions

In [ ]:
from trading_tools.apps.whale_monitor.analyser import (
    hourly_distribution,
    market_breakdown,
    outcome_breakdown,
    side_breakdown,
    summarize_trades,
    trades_to_df,
)

# ── Scalar summary for each wallet ────────────────────────────────────────────
summaries = {
    username: summarize_trades(found[username], trades) for username, trades in whale_trades.items()
}

# Build a single multi-wallet summary DataFrame using to_series()
summary_df = pd.DataFrame([s.to_series() for s in summaries.values()])
display(summary_df)

# ── Breakdowns for a single wallet ────────────────────────────────────────────
username = next(iter(whale_trades))
df = trades_to_df(whale_trades[username])

print(f"\n--- {username}: market breakdown ---")
display(market_breakdown(df).head(10))

print(f"\n--- {username}: outcome breakdown ---")
display(outcome_breakdown(df))

print(f"\n--- {username}: side breakdown ---")
display(side_breakdown(df))

print(f"\n--- {username}: hourly distribution ---")
display(hourly_distribution(df))

In [ ]:
trade_analysis["xdd07070"].market_breakdown